# Run tokenizer on subset of tokenizer data.

Import packages and configure root.

In [ ]:
import numpy as np
import pandas as pd
import sys, subprocess, pickle
from pathlib import Path
!pip install pyfaidx
from huggingface_hub import hf_hub_download

COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")
if COLAB:
    root = Path("/content/mini-gLM")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Download training data and subset to 100k sequences.

In [ ]:
# Get training data from HF
path = hf_hub_download(
    repo_id="eddykang06/mini-gLM-data",
    repo_type="dataset",
    filename="pretraining-train.csv.gz",
)
corpus = pd.read_csv(path, compression = "gzip")

# Select 100k random sequences
rng = np.random.default_rng(111)
idx = rng.choice(np.arange(corpus.shape[0]), size = 100000)
subset_corpus = corpus.iloc[idx]

# Write to csv
subset_corpus.to_csv("tokenizer-training.csv.gz", compression = "gzip")

Train tokenizer and get parameters.

In [ ]:
from src.tokenize import BPETokenizer

# Train tokenizer
sequence_list = subset_corpus["sequence"].to_list()
tokenizer = BPETokenizer()
tokenizer.train(
    sequence_list = sequence_list,
    final_vocab_size = 512
)

# Store and download merge rules and token map
merge_rules = tokenizer.merge_rules
token_to_idx = tokenizer.token_to_idx

with open("merge-rules-512.pkl", "wb") as f:
    pickle.dump(merge_rules, f)

with open("token-map-512.pkl", "wb") as f:
    pickle.dump(token_to_idx, f)